In [1]:
import requests                      # To send HTTP requests
from bs4 import BeautifulSoup        # For parsing HTML content
import pandas as pd                  # For data manipulation
import re                            # For cleaning numeric values


In [ ]:
# ======================================================
# Project: Unified Military Analytics & Comparison Dashboard
# Author : Rutuja Ghodake
# Description:
# Collects global military data from GlobalFirepower.com
# and merges all metrics into a single structured dataset.
# ======================================================

import requests                      # To send HTTP requests
from bs4 import BeautifulSoup        # For parsing HTML content
import pandas as pd                  # For data manipulation
import re                            # For cleaning numeric values(regular expression)

# ------------------------------
# Global Headers
# ------------------------------
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

# -------------------------------------------------------
# Read URLs from text file
# -------------------------------------------------------
def read_links_txt(path):
    links = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("http"):
                links.append(line)
    return links


# -------------------------------------------------------
# Main Scraping Function
# -------------------------------------------------------
def scrape_global_firepower():

    base_url = "https://www.globalfirepower.com/countries-listing.php"
    metric_urls = read_links_txt("All_links.txt")

    # -------------------------------
    # STEP 1: Fetch country list
    # -------------------------------
    response = requests.get(base_url, headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")

    containers = soup.select("div.picTrans.recordsetContainer")

    countries = []
    ranks = []

    for box in containers:
        try:
            country = box.find("span", class_="textWhite textLarge textShadow").text.strip()
            rank = box.find("span", class_="textWhite textLarge textBold").text.strip()
            countries.append(country)
            ranks.append(rank)
        except:
            continue

    df = pd.DataFrame({
        "Country": countries,
        "Rank": ranks
    })

    # -------------------------------
    # STEP 2: Extract all metrics
    # -------------------------------
    for url in metric_urls:
        print(f"Scraping: {url}")

        response = requests.get(url, headers=HEADERS)
        soup = BeautifulSoup(response.text, "html.parser")

        rows = soup.select("div.picTrans.recordsetContainer")

        temp_countries = []
        values = []

        for row in rows:
            try:
                country = row.find("span", class_="textWhite textLarge textShadow").text.strip()
                value = row.find_all("span", class_="textWhite textLarge")[-1].text.strip()
                temp_countries.append(country)
                values.append(value)
            except:
                continue

        column_name = url.split("/")[-1].replace(".php", "")

        temp_df = pd.DataFrame({
            "Country": temp_countries,
            column_name: values
        })

        df = df.merge(temp_df, on="Country", how="left")

    # -------------------------------
    # STEP 3: Clean numeric values
    # -------------------------------
    for col in df.columns[2:]:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.extract(r"(\d+\.?\d*)")[0]
        )

    return df


# -------------------------------------------------------
# RUN SCRIPT
# -------------------------------------------------------
df = scrape_global_firepower()

# Save output
df.to_csv("global_military_data_final.csv", index=False)

# Display results
print("\n✅ Data extraction completed successfully!")
print("Total Countries:", len(df))
print("Total Features:", len(df.columns))
print("\nPreview:\n")
print(df.head())




In [50]:
#Read csv data by using pandas library
import pandas as pd

df = pd.read_csv("Global_military_data_final.csv")

# Preview data
df.head()


,Country,Rank,total-population-by-country,available-military-manpower,manpower-fit-for-military-service,manpower-reaching-military-age-annually,active-military-manpower,active-reserve-military-manpower,manpower-paramilitary,capital-cities-by-total-population,...,natural-gas-production-by-country,natural-gas-consumption-by-country,proven-natural-gas-reserves-by-country,coal-production-by-country,coal-consumption-by-country,proven-coal-reserves-by-country,square-land-area,coastline-coverage,border-coverage,waterway-coverage
0,United States,1,341963408,150463900,124816644,4445524,1328000,799500,0,NaN,...,1029000000000,914301000000,13402000000000,548849000,476044000,248941000000,9833517,19924,12002,41009
1,Russia,2,140820810,69002197,46189226,1267387,1320000,2000000,250000,NaN,...,617830000000,472239000000,47805000000000,508190000,310958000,162166000000,17098242,37653,22407,102000
2,China,3,1415043270,764123366,626864169,19810606,2035000,510000,625000,NaN,...,225341000000,366160000000,6654000000000,4827000000,5313000000,143197000000,9596960,14500,22457,27700
3,India,4,1409128296,662290299,522786598,23955181,1455550,1155000,2527000,NaN,...,33170000000,58867000000,1381000000000,985671000,1200000000,111052000000,3287263,7000,13888,14500
4,South Korea,5,52081799,26040900,21353538,416654,600000,3100000,120000,NaN,...,55127000,59480000000,7079000000,15595000,136413000,326000000,99720,2413,237,1600


In [51]:
#Understand Data Types & Null Values
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 56 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Country                                  145 non-null    object 
 1   Rank                                     145 non-null    int64  
 2   total-population-by-country              145 non-null    int64  
 3   available-military-manpower              145 non-null    int64  
 4   manpower-fit-for-military-service        145 non-null    int64  
 5   manpower-reaching-military-age-annually  145 non-null    int64  
 6   active-military-manpower                 145 non-null    int64  
 7   active-reserve-military-manpower         145 non-null    int64  
 8   manpower-paramilitary                    145 non-null    int64  
 9   capital-cities-by-total-population       3 non-null      float64
 10  aircraft-total                           145 non-n

In [66]:
# Clean Column Names
df.columns = (
    df.columns
    .str.replace("-", "_")
    .str.replace(" ", "_")
    .str.title()
)



In [53]:
# Remove Special Characters (commas, tabs, symbols)

import re  #regular expression

for col in df.columns[2:]:  # Skip Country & Rank
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace(r"\s+", "", regex=True)
    )
df.columns

Index(['Country', 'Rank', 'Total_Population_By_Country',
       'Available_Military_Manpower', 'Manpower_Fit_For_Military_Service',
       'Manpower_Reaching_Military_Age_Annually', 'Active_Military_Manpower',
       'Active_Reserve_Military_Manpower', 'Manpower_Paramilitary',
       'Capital_Cities_By_Total_Population', 'Aircraft_Total',
       'Aircraft_Total_Fighters', 'Aircraft_Total_Attack_Types',
       'Aircraft_Total_Transports', 'Aircraft_Total_Trainers',
       'Aircraft_Total_Special_Mission', 'Aircraft_Total_Tanker_Fleet',
       'Aircraft_Helicopters_Total', 'Aircraft_Helicopters_Attack',
       'Armor_Tanks_Total', 'Armor_Apc_Total',
       'Armor_Self_Propelled_Guns_Total', 'Armor_Towed_Artillery_Total',
       'Armor_Mlrs_Total', 'Navy_Ships', 'Navy_Force_By_Tonnage',
       'Navy_Aircraft_Carriers', 'Navy_Helo_Carriers', 'Navy_Submarines',
       'Navy_Destroyers', 'Navy_Frigates', 'Navy_Corvettes',
       'Navy_Patrol_Coastal_Craft', 'Navy_Mine_Warfare_Craft',
       

In [54]:
print(df.columns)


Index(['Country', 'Rank', 'Total_Population_By_Country',
       'Available_Military_Manpower', 'Manpower_Fit_For_Military_Service',
       'Manpower_Reaching_Military_Age_Annually', 'Active_Military_Manpower',
       'Active_Reserve_Military_Manpower', 'Manpower_Paramilitary',
       'Capital_Cities_By_Total_Population', 'Aircraft_Total',
       'Aircraft_Total_Fighters', 'Aircraft_Total_Attack_Types',
       'Aircraft_Total_Transports', 'Aircraft_Total_Trainers',
       'Aircraft_Total_Special_Mission', 'Aircraft_Total_Tanker_Fleet',
       'Aircraft_Helicopters_Total', 'Aircraft_Helicopters_Attack',
       'Armor_Tanks_Total', 'Armor_Apc_Total',
       'Armor_Self_Propelled_Guns_Total', 'Armor_Towed_Artillery_Total',
       'Armor_Mlrs_Total', 'Navy_Ships', 'Navy_Force_By_Tonnage',
       'Navy_Aircraft_Carriers', 'Navy_Helo_Carriers', 'Navy_Submarines',
       'Navy_Destroyers', 'Navy_Frigates', 'Navy_Corvettes',
       'Navy_Patrol_Coastal_Craft', 'Navy_Mine_Warfare_Craft',
       

In [55]:
# Handle Null Values (NaN)

exclude_column = "Capital_Cities_By_Total_Population"

numeric_cols = df.select_dtypes(include="number").columns

if exclude_column in numeric_cols:
    numeric_cols = numeric_cols.drop(exclude_column)

df[numeric_cols] = df[numeric_cols].fillna(
    df[numeric_cols].median()
)



In [67]:
# Convert Strings to Numbers (string → numeric)
for col in df.columns[2:]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
# df.columns

In [58]:
# Label Encoding (Text → Number)
df["Country_Encoded"] = df["Country"].astype("category").cat.codes


In [ ]:
# Replace missing Navy Force tonnage values with 0
# Reason: Countries without naval tonnage data either do not operate
df["Navy_Force_By_Tonnage"] = df["Navy_Force_By_Tonnage"].fillna(0)

In [68]:
# Final Check
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 57 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Country                                  145 non-null    object 
 1   Rank                                     145 non-null    int64  
 2   Total_Population_By_Country              145 non-null    int64  
 3   Available_Military_Manpower              145 non-null    int64  
 4   Manpower_Fit_For_Military_Service        145 non-null    int64  
 5   Manpower_Reaching_Military_Age_Annually  145 non-null    int64  
 6   Active_Military_Manpower                 145 non-null    int64  
 7   Active_Reserve_Military_Manpower         145 non-null    int64  
 8   Manpower_Paramilitary                    145 non-null    int64  
 9   Capital_Cities_By_Total_Population       3 non-null      float64
 10  Aircraft_Total                           145 non-n

,Country,Rank,Total_Population_By_Country,Available_Military_Manpower,Manpower_Fit_For_Military_Service,Manpower_Reaching_Military_Age_Annually,Active_Military_Manpower,Active_Reserve_Military_Manpower,Manpower_Paramilitary,Capital_Cities_By_Total_Population,...,Natural_Gas_Consumption_By_Country,Proven_Natural_Gas_Reserves_By_Country,Coal_Production_By_Country,Coal_Consumption_By_Country,Proven_Coal_Reserves_By_Country,Square_Land_Area,Coastline_Coverage,Border_Coverage,Waterway_Coverage,Country_Encoded
0,United States,1,341963408,150463900,124816644,4445524,1328000,799500,0,NaN,...,914301000000,13402000000000,548849000,476044000,248941000000,9833517,19924,12002,41009,137
1,Russia,2,140820810,69002197,46189226,1267387,1320000,2000000,250000,NaN,...,472239000000,47805000000000,508190000,310958000,162166000000,17098242,37653,22407,102000,107
2,China,3,1415043270,764123366,626864169,19810606,2035000,510000,625000,NaN,...,366160000000,6654000000000,4827000000,5313000000,143197000000,9596960,14500,22457,27700,28
3,India,4,1409128296,662290299,522786598,23955181,1455550,1155000,2527000,NaN,...,58867000000,1381000000000,985671000,1200000000,111052000000,3287263,7000,13888,14500,53
4,South Korea,5,52081799,26040900,21353538,416654,600000,3100000,120000,NaN,...,59480000000,7079000000,15595000,136413000,326000000,99720,2413,237,1600,117


In [65]:
# Save final clean dataset
df.to_csv("Global_Military_Data_Cleaned.csv", index=False)

print("✅ Final cleaned dataset saved successfully")


✅ Final cleaned dataset saved successfully
